In [32]:
#*********************************
# Library
#*********************************
import pynmea2
import mplleaflet
import pyproj 
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import gmplot as gp
import webbrowser as web

In [33]:
from pyproj import CRS, Transformer

def utm_to_latlon(utm_x_list, utm_y_list, zone_number, zone_letter):
    # zone_letter: 'L', 'M' = sur ; 'N'..'X' = norte
    is_northern = (zone_letter.upper() >= 'N')
    epsg = (32600 + zone_number) if is_northern else (32700 + zone_number)

    transformer = Transformer.from_crs(CRS.from_epsg(epsg), CRS.from_epsg(4326), always_xy=True)
    lon_list, lat_list = transformer.transform(utm_x_list, utm_y_list)
    return list(lat_list), list(lon_list)

In [34]:
#*********************************
# Get Coordinates
#*********************************

def get_coordinates(nmea_data):
    
    coordinates_data = []
    for message_bytes in nmea_data.readlines():    
        try:
            message = message_bytes.decode("utf-8").replace("\n", "").replace("\r", "")
            parsed_message = pynmea2.parse(message)
        except:
            continue

        cga_data = {}
        
        # process only GGA messages
        if parsed_message.sentence_type == "GGA":
            for attr in ["timestamp", "latitude", "longitude", "latitude", "horizontal_dil", "num_sats", "gps_qual"]:
                cga_data[attr] = getattr(parsed_message, attr)
            coordinates_data.append(cga_data)
    
    return coordinates_data


In [35]:
def get_log(log_path):
    df_log = pd.read_csv(log_path, dtype=str)
    for column in df_log:
        df_log[column] = df_log[column].apply(lambda x: int.from_bytes(bytes.fromhex(x), byteorder="little", signed=True))
    return df_log

In [ ]:
# df_log_basico = get_log('pruebas_27-03/ruta2-1_ts.csv')
# df_log_basico

,ts,utm_x,utm_y,z,utm_zone_n,utm_zone_l,heading,inx_x,inx_y,inx_z,inx_yaw
0,1546950776,27701193,868651487,11898,18,76,34560,-25,71,-25,-8860
1,1546950877,27701184,868651521,11898,18,76,34560,-53,92,-53,-8860
2,1546950976,27701176,868651555,11898,18,76,34559,-81,112,-81,-8861
3,1546951075,27701167,868651588,11898,18,76,34559,-109,133,-109,-8861
4,1546951178,27701158,868651622,11898,18,76,34559,-137,154,-137,-8861
...,...,...,...,...,...,...,...,...,...,...,...
1156,1547066560,27699276,868659791,11898,18,76,34557,-28706,10951,-31967,-8863
1157,1547066660,27699268,868659825,11898,18,76,34557,-28727,10923,-31994,-8863
1158,1547066759,27699259,868659858,11898,18,76,34557,-28747,10895,-32022,-8863
1159,1547066860,27699250,868659892,11898,18,76,34557,-28767,10867,-32050,-8863


In [37]:
df_log_tactico = get_log('pruebas_27-03/ruta2-1_ts.csv')
df_log_tactico

,ts,utm_x,utm_y,z,utm_zone_n,utm_zone_l,heading,inx_x,inx_y,inx_z,inx_yaw
0,1546950776,27701193,868651487,11898,18,76,34560,-25,71,-25,-8860
1,1546950877,27701184,868651521,11898,18,76,34560,-53,92,-53,-8860
2,1546950976,27701176,868651555,11898,18,76,34559,-81,112,-81,-8861
3,1546951075,27701167,868651588,11898,18,76,34559,-109,133,-109,-8861
4,1546951178,27701158,868651622,11898,18,76,34559,-137,154,-137,-8861
...,...,...,...,...,...,...,...,...,...,...,...
1156,1547066560,27699276,868659791,11898,18,76,34557,-28706,10951,-31967,-8863
1157,1547066660,27699268,868659825,11898,18,76,34557,-28727,10923,-31994,-8863
1158,1547066759,27699259,868659858,11898,18,76,34557,-28747,10895,-32022,-8863
1159,1547066860,27699250,868659892,11898,18,76,34557,-28767,10867,-32050,-8863


In [38]:
coordinates_data = []

# Read File:
# ------------
nmea_data = open("CONTINUACIÓN_RUTAS_GIANMARCO/CONT_RUTAS/RUTA_2/ruta_gnss.nmea", "rb")

# Get coordinates
#-----------------
coordinates_data = get_coordinates(nmea_data)

# Total data
#------------
df = pd.DataFrame(coordinates_data)
df_latitude  = df['latitude'].tolist()
df_longitude = df['longitude'].tolist()

# Graph coordinates
# -----------------
ref_lat  = -12.103812
ref_long = -77.020169
zoom     = 16
gmap= gp.GoogleMapPlotter(ref_lat,ref_long,zoom)
gmap.scatter(df_latitude,df_longitude,'#FF0000',size=1,marker=False)
gmap.plot(df_latitude,df_longitude,'#AA0000',edge_width=3.0)

zone_number = int(df_log_basico["utm_zone_n"].iloc[0])
zone_l_raw = int(df_log_basico["utm_zone_l"].iloc[0])
zone_letter = chr(zone_l_raw)   # 76 -> 'L'

df_log_basico["utm_x_m"] = df_log_basico["utm_x"].astype(float) / 100.0
df_log_basico["utm_y_m"] = df_log_basico["utm_y"].astype(float) / 100.0
log_ins_b_lat, log_ins_b_lon = utm_to_latlon(
    df_log_basico["utm_x_m"].tolist(),
    df_log_basico["utm_y_m"].tolist(),
    zone_number,
    zone_letter
)

gmap.scatter(log_ins_b_lat, log_ins_b_lon, "#FFA500", size=1, marker=False)
gmap.plot(log_ins_b_lat, log_ins_b_lon, '#FFA500', edge_width=2.0)

df_log_tactico["utm_x_m"] = df_log_tactico["utm_x"].astype(float) / 100.0
df_log_tactico["utm_y_m"] = df_log_tactico["utm_y"].astype(float) / 100.0
log_ins_t_lat, log_ins_t_lon = utm_to_latlon(
    df_log_tactico["utm_x_m"].tolist(),
    df_log_tactico["utm_y_m"].tolist(),
    zone_number,
    zone_letter
)

#print(zone_letter, zone_number, df_log_basico["utm_x"].iloc[1], log_ins_b_lat[1], log_ins_b_lon[1]);

gmap.scatter(log_ins_t_lat, log_ins_t_lon, "#0000FF", size=1, marker=False)
gmap.plot(log_ins_t_lat, log_ins_t_lon, '#0000AA', edge_width=2.0)

gmap.draw("prueba_14.html")    # dibuja la trayectoria y lo guarda en un archivo. 

# Abrir Archivo:
# ---------------
#browser = web.get(using='windows-default')
#web.open_new("habish4.html")